# L3 — Sensibilidade direcional contínua na camada 11

Para cada direção unitária \(v\), comparamos o TCAV binário \(\Pr[g_i^\top v>0]\) com a derivada direcional média \(\operatorname{mean}_i(g_i^\top v)\), na escala de probabilidade e na escala de log-odds. Usamos a separação \(g_i=p_i(1-p_i)\nabla_h\mathrm{logit}(p_i)\).

A primeira coorte de cada série fixa um eixo comum (média das direções unitárias dos gradientes de log-odds) e as direções de controle. Aplicamos o mesmo eixo e os mesmos controles nos anos seguintes. A projeção de cada gradiente nesse eixo dá uma aproximação de posto 1. Comparamos os escores médios por direção e a energia dos gradientes retida. Os controles incluem direções isotrópicas, a primeira componente principal e amostras no subespaço das cinco primeiras componentes e no seu complemento. Medimos as duas cartas do Trabalho 8, coordenadas originais e padronização congelada de referência.

Para o SUS histórico de referência 2011 acrescentamos os CAVs congelados compatíveis do Trabalho 2. Eles foram ajustados no sistema histórico, e são avaliados aqui nos anos disponíveis no Trabalho 8. Os CAVs e controles respondem a perguntas distintas. Os controles não foram ajustados para representar conceitos. Para a leitura longitudinal, este é um diagnóstico descritivo nas coortes salvas, com o eixo calculado em sua coorte de referência.

In [1]:
from hashlib import sha256
from pathlib import Path

import numpy as np
import pandas as pd

import flchain_probe as probe
import tcav_geometry as geometry
from concept_study.loading import load_cav_directions

SUS = Path("analysis_outputs/sus_layer_compression/layer_arrays.npz")
FLCHAIN = Path("analysis_outputs/flchain_layer_compression/layer_arrays.npz")
FLCHAIN_CSV = Path("data/flchain/free_light_chain_mortality.csv")
CAVS = Path("stats/l3_l4_inputs/cav_directions.npz")
OUT = Path("analysis_outputs/l3_continuous_tcav")
LAYER = 11
DRAWS = 2000
TOP = 5
SEED = 927
SUS_SERIES = ("same_year_consistent", "same_year_historical", "prospective_consistent")


def layer_data(archive, prefix=""):
    embedding = archive[prefix + "target_embedding"][:, LAYER].astype(float)
    gradient = archive[prefix + "target_gradient"][:, LAYER].astype(float)
    probability = archive[prefix + "probability"].astype(float)
    scale = archive[prefix + "scale_target"][LAYER].astype(float)
    assert embedding.shape == gradient.shape == (len(probability), len(scale))
    assert np.isfinite(embedding).all() and np.isfinite(gradient).all()
    assert np.isfinite(scale).all() and (scale > 0).all()
    assert ((0 <= probability) & (probability <= 1)).all()
    return embedding, gradient, probability, scale


series = {}
with np.load(SUS) as archive:
    for name in SUS_SERIES:
        prefix = name + "__"
        series[name] = (*layer_data(archive, prefix), archive[prefix + "year"], 2011)

with np.load(FLCHAIN) as archive:
    frame = probe.load_flchain_dataframe(FLCHAIN_CSV, expected_sha256=probe.EXPECTED_CSV_SHA256)
    kept, _ = probe.build_flchain_target(frame, 1095)
    indices = archive["eval_row_index"]
    assert np.array_equal(kept["subject_id"].to_numpy()[indices], archive["subject_id"])
    years = kept["sample.yr"].to_numpy()[indices]
    assert np.array_equal(years == 1997, archive["eval_stratum"] == "1997_eval")
    series["flchain_h1095"] = (*layer_data(archive), years, 1997)

cavs, cav_counts = load_cav_directions(CAVS)
cav_sets = []
for (estimator, key), matrix in sorted(cavs.items()):
    if not key.startswith("last_3/2011/"):
        continue
    keep = np.isfinite(matrix).all(axis=1)
    assert keep.sum() == cav_counts[estimator, key][1]
    fits = matrix[keep].astype(float)
    np.testing.assert_allclose(np.linalg.norm(fits, axis=1), 1, rtol=1e-6)
    cav_sets.append((f"cav_{estimator}", fits, key, np.flatnonzero(keep) + 1001))

assert cav_sets
assert {name for name, *_ in cav_sets} == {"cav_all_negatives", "cav_legacy"}
print("Famílias de CAVs por estimador:", pd.Series([name for name, *_ in cav_sets]).value_counts().to_dict())

Famílias de CAVs por estimador: {'cav_all_negatives': 14, 'cav_legacy': 14}


In [2]:
def score_directions(gradient, directions, axis):
    projection = gradient @ axis
    sensitivities = gradient @ directions.T
    approximate = projection[:, None] * (directions @ axis)
    mean = sensitivities.mean(axis=0)
    mean_approximate = approximate.mean(axis=0)
    residual = gradient - projection[:, None] * axis
    np.testing.assert_allclose(
        mean, gradient.mean(axis=0) @ directions.T, rtol=1e-9, atol=1e-12
    )
    return {
        "sign_tcav": (sensitivities > 0).mean(axis=0),
        "mean_directional_derivative": mean,
        "mean_absolute_directional_derivative": np.abs(sensitivities).mean(axis=0),
        "rank_one_mean_derivative": mean_approximate,
        "direction_axis_cosine": directions @ axis,
        "rank_one_gradient_energy": 1 - np.sum(residual ** 2) / np.sum(gradient ** 2),
    }


records = []
for name, (embedding, gradient, probability, scale, years, reference) in series.items():
    valid = (probability > 0) & (probability < 1)
    raw_log_gradient = geometry.log_odds_gradients(gradient, probability)
    assert np.isfinite(raw_log_gradient[valid]).all()
    ref = (years == reference) & valid
    groups = {str(year): years == year for year in sorted(set(years))}
    groups["later_pooled"] = years > reference
    for chart, factor in (("raw", np.ones_like(scale)), ("standardized", scale)):
        log_gradient, prob_gradient = raw_log_gradient * factor, gradient * factor
        reference_axis = geometry.coherence(log_gradient[ref])["mean_direction"]
        np.testing.assert_allclose(np.linalg.norm(reference_axis), 1, atol=1e-12)
        _, pcs, _ = geometry.principal_axes(embedding[ref] / factor)
        rng = np.random.default_rng(SEED)
        direction_sets = [
            (family, geometry.random_directions(rng, DRAWS, pcs, TOP, family), "", np.arange(DRAWS))
            for family in geometry.DIRECTION_FAMILIES
        ]
        direction_sets.append(("pc1", pcs[:, :1].T, "", np.array([0])))
        if name == "same_year_historical" and chart == "raw":
            direction_sets += cav_sets
        for cohort, mask in groups.items():
            rows = mask & valid
            for output, g in (("probability", prob_gradient), ("log_odds", log_gradient)):
                for family, directions, concept, fit_ids in direction_sets:
                    values = score_directions(g[rows], directions, reference_axis)
                    records.append(pd.DataFrame({
                        **values, "series": name, "cohort": cohort, "chart": chart,
                        "output": output, "family": family, "concept": concept,
                        "direction_id": fit_ids, "n_rows": int(rows.sum()),
                        "excluded_boundary_probabilities": int((mask & ~valid).sum()),
                    }))

directions = pd.concat(records, ignore_index=True)
assert directions["sign_tcav"].between(0, 1).all()
assert directions["rank_one_gradient_energy"].between(0, 1).all()
assert len(directions.drop_duplicates(["series", "cohort", "chart", "output", "family", "concept", "direction_id"])) == len(directions)

In [3]:
def summarize(frame):
    score = frame["mean_directional_derivative"].to_numpy()
    fit = frame["rank_one_mean_derivative"].to_numpy()
    sign = frame["sign_tcav"].to_numpy()
    unique_concepts = frame["concept"].replace("", np.nan).nunique()
    return {
        "rows": int(frame["n_rows"].iloc[0]),
        "directions": len(frame),
        "concepts": unique_concepts,
        "saturated_sign_share": np.mean((sign == 0) | (sign == 1)),
        "median_absolute_score": np.median(np.abs(score)),
        "p95_absolute_score": np.quantile(np.abs(score), 0.95),
        "rank_one_gradient_energy": frame["rank_one_gradient_energy"].iloc[0],
        "rank_one_score_relative_rmse": np.linalg.norm(score - fit) / np.linalg.norm(score),
        "rank_one_score_correlation": np.corrcoef(score, fit)[0, 1] if len(score) > 1 else np.nan,
        "rank_one_score_sign_agreement": np.mean((score > 0) == (fit > 0)),
    }


keys = ["series", "cohort", "chart", "output", "family"]
summaries = pd.DataFrame([
    {**dict(zip(keys, group)), **summarize(frame)}
    for group, frame in directions.groupby(keys, sort=True)
])
OUT.mkdir(parents=True, exist_ok=True)
directions.to_csv(OUT / "direction_scores.csv.gz", index=False,
                  compression={"method": "gzip", "mtime": 0})
summaries.to_csv(OUT / "summary.csv", index=False)
pd.DataFrame([{"path": str(path), "sha256": sha256(path.read_bytes()).hexdigest()}
              for path in (SUS, FLCHAIN, FLCHAIN_CSV, CAVS)]).to_csv(OUT / "inputs.csv", index=False)

display(summaries[
    (summaries["chart"] == "raw") & (summaries["family"].isin(["isotropic", "cav_all_negatives"])) &
    (summaries["cohort"] == "later_pooled")
].round(5))

/tmp/ipykernel_149354/218822988.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  unique_concepts = frame["concept"].replace("", np.nan).nunique()


,series,cohort,chart,output,family,rows,directions,concepts,saturated_sign_share,median_absolute_score,p95_absolute_score,rank_one_gradient_energy,rank_one_score_relative_rmse,rank_one_score_correlation,rank_one_score_sign_agreement
113,flchain_h1095,later_pooled,raw,log_odds,isotropic,1673,2000,0,0.53300,0.04489,0.12316,0.96852,0.06668,0.99778,0.9815
117,flchain_h1095,later_pooled,raw,probability,isotropic,1673,2000,0,0.53300,0.00155,0.00437,0.81096,0.18750,0.98227,0.9430
177,prospective_consistent,later_pooled,raw,log_odds,isotropic,589,2000,0,0.85700,0.03870,0.10578,0.99007,0.06098,0.99814,0.9830
181,prospective_consistent,later_pooled,raw,probability,isotropic,589,2000,0,0.85700,0.00218,0.00608,0.99031,0.06471,0.99791,0.9840
241,same_year_consistent,later_pooled,raw,log_odds,isotropic,660,2000,0,0.63350,0.04111,0.11295,0.94238,0.07131,0.99745,0.9790
245,same_year_consistent,later_pooled,raw,probability,isotropic,660,2000,0,0.63350,0.00215,0.00603,0.81834,0.23754,0.97142,0.9180
316,same_year_historical,later_pooled,raw,log_odds,cav_all_negatives,660,700,14,0.98571,0.27041,0.50277,0.94806,0.00953,0.99996,1.0000
319,same_year_historical,later_pooled,raw,log_odds,isotropic,660,2000,0,0.64350,0.03786,0.10395,0.94806,0.04748,0.99887,0.9885
322,same_year_historical,later_pooled,raw,probability,cav_all_negatives,660,700,14,0.98571,0.01733,0.03274,0.81902,0.06860,0.99946,1.0000
325,same_year_historical,later_pooled,raw,probability,isotropic,660,2000,0,0.64350,0.00233,0.00654,0.81902,0.23633,0.97171,0.9210


## CAVs congelados em todas as referências do Trabalho 2

O script `l3_work2_cav_summary.py` usa as sensibilidades por paciente já calculadas na Fiji. Ele exporta somente escores por família, ajuste de CAV e ano de entrada até 2014. Para cada ajuste, calculamos o TCAV por sinal e as derivadas direcionais média e absoluta em probabilidade e em log-odds. A comparação com o eixo de posto 1 acima permanece no diagnóstico do Trabalho 8, em que temos os vetores completos de gradiente e um eixo de referência comum.

In [4]:
work2_path = OUT / "work2_cav_scores.csv.gz"
work2 = pd.read_csv(work2_path)
assert work2["concept"].nunique() == 61
assert work2["reference_year"].nunique() == 7
assert work2["tcav_sign"].between(0, 1).all()
assert not work2.duplicated(
    ["build", "reference_year", "input_year", "concept", "estimator", "output", "fit_index"]
).any()
assert np.isfinite(work2["mean_directional_derivative"]).all()
assert (work2["input_year"] <= 2014).all()

work2["fully_saturated"] = work2["tcav_sign"].isin([0, 1])
work2["absolute_mean_score"] = work2["mean_directional_derivative"].abs()
work2["period"] = np.where(work2["input_year"] == work2["reference_year"], "reference", "later")
cav_summary = work2.groupby(["estimator", "output", "period"], as_index=False).agg(
    concepts=("concept", "nunique"),
    fit_year_pairs=("tcav_sign", "size"),
    saturated_sign_share=("fully_saturated", "mean"),
    p05_abs_mean=("absolute_mean_score", lambda values: values.quantile(0.05)),
    median_abs_mean=("absolute_mean_score", "median"),
    p95_abs_mean=("absolute_mean_score", lambda values: values.quantile(0.95)),
)
cav_summary.to_csv(OUT / "work2_cav_summary.csv", index=False)
print("Referências do Trabalho 2:", sorted(work2["reference_year"].unique()))
print("Hash do índice de origem:", (OUT / "work2_patient_table_index_hash.txt").read_text().strip())
display(cav_summary.round(5))

Referências do Trabalho 2: [2007, 2008, 2009, 2010, 2011, 2012, 2013]
Hash do índice de origem: 46a5434c39671ef4


,estimator,output,period,concepts,fit_year_pairs,saturated_sign_share,p05_abs_mean,median_abs_mean,p95_abs_mean
0,all_negatives,log_odds,later,61,10244,0.98907,0.06046,0.26882,0.49497
1,all_negatives,log_odds,reference,61,3003,0.98102,0.04886,0.31992,0.53997
2,all_negatives,probability,later,61,10244,0.98907,0.00461,0.01742,0.03108
3,all_negatives,probability,reference,61,3003,0.98102,0.00279,0.01697,0.02915
4,legacy,log_odds,later,61,10238,0.97578,0.04970,0.26858,0.49247
5,legacy,log_odds,reference,61,3002,0.97035,0.03943,0.32068,0.53810
6,legacy,probability,later,61,10238,0.97578,0.00360,0.01746,0.03105
7,legacy,probability,reference,61,3002,0.97035,0.00239,0.01697,0.02907


## Variação temporal comum aos conceitos

Para cada conceito e ano, tomamos a média da derivada direcional sobre os ajustes de CAV e usamos o logaritmo do valor absoluto. Subtraímos a média de cada conceito ao longo dos anos e medimos que parcela dessa variação temporal é explicada pela média de todos os conceitos do mesmo modelo e ano. Uma parcela alta indica que o escore contínuo muda junto para todos os conceitos, acompanhando a escala geral dos gradientes.

In [5]:
concept_year = work2.groupby(
    ["estimator", "output", "reference_year", "concept", "input_year"], as_index=False
)["mean_directional_derivative"].mean()
concept_year = concept_year[concept_year["mean_directional_derivative"] != 0].copy()
concept_year["log_magnitude"] = np.log(concept_year["mean_directional_derivative"].abs())
by_concept = ["estimator", "output", "reference_year", "concept"]
concept_year["within_concept"] = (
    concept_year["log_magnitude"] - concept_year.groupby(by_concept)["log_magnitude"].transform("mean")
)
concept_year["model_year_mean"] = concept_year.groupby(
    ["estimator", "output", "reference_year", "input_year"]
)["within_concept"].transform("mean")


def shared_share(frame):
    residual = frame["within_concept"] - frame["model_year_mean"]
    return 1 - np.sum(residual ** 2) / np.sum(frame["within_concept"] ** 2)


temporal_share = pd.DataFrame([
    {"estimator": estimator, "output": output, "concepts": frame["concept"].nunique(),
     "concept_years": len(frame), "shared_temporal_share": shared_share(frame)}
    for (estimator, output), frame in concept_year.groupby(["estimator", "output"])
])
temporal_share.to_csv(OUT / "work2_cav_temporal_share.csv", index=False)
display(temporal_share.round(4))

,estimator,output,concepts,concept_years,shared_temporal_share
0,all_negatives,log_odds,61,271,0.9448
1,all_negatives,probability,61,271,0.9133
2,legacy,log_odds,61,271,0.9332
3,legacy,probability,61,271,0.8993
